# Window functions

Sorting rows within each group and walking through them in order: running totals, rankings,
comparisons against the previous row. This is the engine room of feature pipelines and any
"compared to last time" metric, and it is genuinely heavy work.

**W4** is one of the few places the dialects disagree. Subtracting two dates gives a number of
days in DuckDB and an interval in Spark, so it needs engine-specific SQL.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from bench import config as C, datagen, engines, report
from bench.harness import Case, Bench

paths = datagen.paths(C.MAIN_SIZE)
duck  = engines.get_duckdb()
spark = engines.get_spark()
engines.attach(duck, spark, paths)
bench = Bench(duck, spark, "04_windows", C.MAIN_SIZE)
print(f"Ready. {C.human(C.MAIN_SIZE)} sales rows. "
      f"Both engines have {C.ENGINE_MEMORY_MB} MB and {C.plural(C.ENGINE_THREADS, 'thread')}.")


In [ ]:
SQL_W1 = """
SELECT segment, count(*) AS n FROM (
  SELECT c.segment, s.customer_id, sum(s.amount) AS spend,
         rank() OVER (PARTITION BY c.segment ORDER BY sum(s.amount) DESC) AS rk
  FROM sales s JOIN customers c ON c.customer_id = s.customer_id
  GROUP BY 1,2
) t WHERE rk <= 100 GROUP BY 1 ORDER BY 1
"""

_, out, _ = bench.run(Case("W1", "Rank customers by spend", "Windows", sql=SQL_W1.strip()))
display(out.head())


In [ ]:
SQL_W2 = """
SELECT round(avg(running),2) AS avg_running, max(running) AS max_running FROM (
  SELECT sum(amount) OVER (PARTITION BY customer_id ORDER BY sale_ts
                           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running
  FROM sales WHERE customer_id IS NOT NULL AND amount IS NOT NULL
) t
"""

_, out, _ = bench.run(Case("W2", "Running total per customer", "Windows", sql=SQL_W2.strip()))
display(out.head())


In [ ]:
SQL_W3 = """
SELECT round(avg(roll),2) AS avg_roll FROM (
  SELECT avg(amount) OVER (PARTITION BY customer_id ORDER BY sale_ts
                           ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS roll
  FROM sales WHERE customer_id IS NOT NULL AND amount IS NOT NULL
) t
"""

_, out, _ = bench.run(Case("W3", "Rolling average of last 5 orders", "Windows", sql=SQL_W3.strip()))
display(out.head())


In [ ]:
SQL_W4_DUCK = """
SELECT count(*) AS n, round(avg(gap_days),2) AS avg_gap_days FROM (
  SELECT date_diff('day', lag(sale_date) OVER (PARTITION BY customer_id ORDER BY sale_date),
                   sale_date) AS gap_days
  FROM sales WHERE customer_id IS NOT NULL
) t WHERE gap_days IS NOT NULL
"""

SQL_W4_SPARK = """
SELECT count(*) AS n, round(avg(gap_days),2) AS avg_gap_days FROM (
  SELECT datediff(sale_date,
                  lag(sale_date) OVER (PARTITION BY customer_id ORDER BY sale_date)) AS gap_days
  FROM sales WHERE customer_id IS NOT NULL
) t WHERE gap_days IS NOT NULL
"""

_, out, _ = bench.run(Case("W4", "Gap between consecutive orders", "Windows", duck_sql=SQL_W4_DUCK.strip(), spark_sql=SQL_W4_SPARK.strip()))
display(out.head())


In [ ]:
SQL_W5 = """
SELECT count(*) AS customers, round(avg(first_amt),2) AS avg_first,
       round(avg(last_amt),2) AS avg_last FROM (
  SELECT DISTINCT customer_id,
    first_value(amount) OVER (PARTITION BY customer_id ORDER BY sale_ts
                              ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS first_amt,
    last_value(amount)  OVER (PARTITION BY customer_id ORDER BY sale_ts
                              ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS last_amt
  FROM sales WHERE customer_id IS NOT NULL AND amount IS NOT NULL
) t
"""

_, out, _ = bench.run(Case("W5", "First and last order per customer", "Windows", sql=SQL_W5.strip()))
display(out.head())


## Results for this notebook

`Same SQL?` tells you whether both engines ran the *identical* SQL string. Where it says no, the two dialects genuinely differ and the case is written twice.

`Same answer?` is the check that matters: a fast wrong answer is worth nothing.

In [ ]:
report.headline(bench.table())
print()
display(report.results_table(bench.table()))
report.times_chart(bench.table())
bench.save()
engines.stop_spark()
